# 04. Combined uncertainty analysis

Combine evidence- and answer-stage uncertainty and evaluate whether they identify incorrect representative answers.

In [1]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import requests
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from openai import OpenAI

## 1. Configuration

In [38]:
def find_project_root():
    cwd = Path.cwd().resolve()
    for path in [cwd] + list(cwd.parents):
        if (path / "outputs").exists():
            return path
    return cwd

PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PROCESSED_DIR = OUTPUT_DIR / "processed_data"

SAMPLE_MODE = False
RUN_NAME = "sample" if SAMPLE_MODE else "full"

EVIDENCE_DIR = OUTPUT_DIR / "evidence_selection" / RUN_NAME
ANSWER_DIR = OUTPUT_DIR / "answer_generation" / RUN_NAME
ANALYSIS_DIR = OUTPUT_DIR / "combined_analysis" / RUN_NAME
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

JUDGE_BACKEND = "ollama" if SAMPLE_MODE else "vllm"

OLLAMA_JUDGE_MODEL = "gemma3:12b"
VLLM_JUDGE_MODEL = "meta-llama/Llama-3.3-70B-Instruct"

JUDGE_MODEL = (
    OLLAMA_JUDGE_MODEL
    if JUDGE_BACKEND == "ollama"
    else VLLM_JUDGE_MODEL
)

VLLM_BASE_URL = "http://localhost:8000/v1"

JUDGE_TEMPERATURE = 0.0
JUDGE_NUM_CTX = 8192

JUDGE_PATH = ANALYSIS_DIR / "judge_results.jsonl"
CASE_PATH = ANALYSIS_DIR / "case_analysis.csv"
CORRELATION_PATH = ANALYSIS_DIR / "correlations.csv"
PROFILE_PATH = ANALYSIS_DIR / "joint_profile_summary.csv"
PREDICTION_PATH = ANALYSIS_DIR / "prediction_metrics.csv"
RISK_PATH = ANALYSIS_DIR / "risk_coverage.csv"
SELECTIVE_PATH = ANALYSIS_DIR / "selective_review.csv"
HELDOUT_DIR = OUTPUT_DIR / "archehr_heldout"
HELDOUT_PATH = HELDOUT_DIR / "case_analysis.csv"


RUN_BOOTSTRAP = not SAMPLE_MODE
N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 42
BOOTSTRAP_ALPHA = 0.05

BOOTSTRAP_METRICS_PATH = ANALYSIS_DIR / "bootstrap_metrics.csv"
BOOTSTRAP_COMPARISONS_PATH = (
    ANALYSIS_DIR / "bootstrap_comparisons.csv"
)

## 2. Shared functions

In [3]:
def load_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def append_jsonl(record, path):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def format_references(reference_answers):
    return "\n".join(
        f"Reference {i}: {answer}"
        for i, answer in enumerate(reference_answers, 1)
    )

def build_judge_prompt(question, reference_answers, answer):
    return f'''
Evaluate the correctness of the generated biomedical answer.

Question:
{question}

Expert reference answer(s):
{format_references(reference_answers)}

Generated answer:
{answer}

Mark the generated answer as correct only if it is substantively correct
and sufficiently complete to answer the question.

Do not penalise paraphrasing or concise wording.
For a multi-part question, omitting a major requested part is incorrect.
A major contradiction or unsupported claim is incorrect.
The generated answer does not need to match every reference exactly.

Return only JSON:
{{"correct": true, "reason": "brief reason"}}
'''.strip()

def call_ollama(prompt, temperature=0.0):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": JUDGE_MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": temperature,
                "top_p": 0.9,
                "num_predict": 128,
                "num_ctx": JUDGE_NUM_CTX,
            },
        },
        timeout=600,
    )
    response.raise_for_status()

    return response.json()["response"].strip()


def call_vllm(prompt, temperature=0.0):
    response = requests.post(
        f"{VLLM_BASE_URL}/chat/completions",
        json={
            "model": JUDGE_MODEL,
            "messages": [
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            "temperature": temperature,
            "top_p": 0.9,
            "max_completion_tokens": 128,
            "stream": False,
        },
        timeout=600,
    )
    response.raise_for_status()

    return (
        response.json()["choices"][0]["message"]["content"]
        .strip()
    )


def call_judge_model(prompt, temperature=0.0):
    if JUDGE_BACKEND == "ollama":
        return call_ollama(
            prompt,
            temperature=temperature,
        )

    if JUDGE_BACKEND == "vllm":
        return call_vllm(
            prompt,
            temperature=temperature,
        )

    raise ValueError(
        f"Unknown judge backend: {JUDGE_BACKEND}"
    )


def check_judge_server():
    if JUDGE_BACKEND == "ollama":
        url = "http://localhost:11434/"
    else:
        url = f"{VLLM_BASE_URL}/models"

    response = requests.get(
        url,
        timeout=10,
    )
    response.raise_for_status()


def parse_judgement(text):
    text = text.replace("```json", "").replace("```", "").strip()

    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*?\}", text, flags=re.DOTALL)
        if match is None:
            return None
        try:
            data = json.loads(match.group(0))
        except json.JSONDecodeError:
            return None

    correct = data.get("correct")
    if isinstance(correct, bool):
        value = correct
    elif isinstance(correct, int) and correct in (0, 1):
        value = bool(correct)
    else:
        return None

    return {
        "correct": value,
        "reason": str(data.get("reason", "")).strip(),
    }

def judge_answer(question, reference_answers, answer):
    prompt = build_judge_prompt(
        question,
        reference_answers,
        answer,
    )

    for _ in range(2):
        result = parse_judgement(
            call_judge_model(
                prompt,
                temperature=JUDGE_TEMPERATURE,
            )
        )
        if result is not None:
            return result

    raise ValueError("Could not parse judge response.")

def representative_answer(group):
    group = group.sort_values("run_id").copy()
    group["cluster_label"] = group["cluster_label"].astype(int)

    counts = group["cluster_label"].value_counts()
    largest = counts.max()
    tied_labels = set(counts[counts == largest].index.tolist())

    # If clusters tie, use the cluster containing the earliest run.
    dominant_label = int(
        group[group["cluster_label"].isin(tied_labels)]
        .iloc[0]["cluster_label"]
    )

    dominant = group[
        group["cluster_label"] == dominant_label
    ].sort_values("run_id")

    row = dominant.iloc[0]

    return {
        "representative_answer": row["answer"],
        "representative_run_id": int(row["run_id"]),
        "representative_cluster_size": len(dominant),
    }

def safe_spearman(x, y):
    data = pd.DataFrame({"x": x, "y": y}).dropna()

    if (
        len(data) < 3
        or data["x"].nunique() < 2
        or data["y"].nunique() < 2
    ):
        return np.nan, np.nan, len(data)

    rho, p_value = spearmanr(data["x"], data["y"])
    return float(rho), float(p_value), len(data)

def safe_auroc(y_true, scores):
    data = pd.DataFrame({
        "y": y_true,
        "score": scores,
    }).dropna()

    if len(data) < 2 or data["y"].nunique() < 2:
        return np.nan

    return float(
        roc_auc_score(
            data["y"].astype(int),
            data["score"],
        )
    )

# Calculate risk-coverage curve with tie handling
def compute_risk_coverage(group, signal):

    data = (
        group[[signal, "answer_error"]]
        .dropna()
        .copy()
    )

    # Group cases with the same uncertainty score
    tie_groups = (
        data.groupby(signal, sort=True)["answer_error"]
        .agg(["size", "sum"])
        .reset_index()
    )

    rows = []

    retained_before = 0
    errors_before = 0.0
    n = len(data)

    for row in tie_groups.itertuples(index=False):

        tie_n = row.size
        tie_errors = row.sum
        tie_error_rate = tie_errors / tie_n

        # Use expected error within each tied group
        for r in range(1, tie_n + 1):

            n_retained = retained_before + r

            expected_errors = (
                errors_before
                + r * tie_error_rate
            )

            error_rate = (
                expected_errors / n_retained
            )

            rows.append({
                "coverage": n_retained / n,
                "n_retained": n_retained,
                "retained_error_rate": error_rate,
                "retained_correctness": 1.0 - error_rate,
            })

        retained_before += tie_n
        errors_before += tie_errors

    return pd.DataFrame(rows)


# Report risk at selected coverage levels
def selective_review(
    group,
    signal,
    targets=(1.0, 0.9, 0.8, 0.7, 0.6),
):

    curve = compute_risk_coverage(
        group,
        signal,
    )

    rows = []
    n = len(curve)

    for target in targets:

        k = max(
            1,
            int(np.ceil(target * n)),
        )

        row = curve.iloc[k - 1]

        rows.append({
            "target_coverage": target,
            "coverage": row["coverage"],
            "n_retained": int(row["n_retained"]),
            "retained_error_rate": row["retained_error_rate"],
            "retained_correctness": row["retained_correctness"],
        })

    return pd.DataFrame(rows)

In [4]:
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
)


def call_judge(prompt):
    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        temperature=0.0,
    )

    return response.choices[0].message.content.strip()

## 3. Load previous outputs

In [5]:
evidence_summary = pd.read_csv(
    EVIDENCE_DIR / "evidence_summary.csv"
)

answer_summary = pd.read_csv(
    ANSWER_DIR / "answer_summary.csv"
)

answer_runs = pd.read_csv(
    ANSWER_DIR / "answer_runs_with_clusters.csv"
)

analysis = evidence_summary.merge(
    answer_summary,
    on=["analysis_set", "case_id"],
    how="inner",
    validate="one_to_one",
)

current_keys = set(
    zip(
        analysis["analysis_set"],
        analysis["case_id"],
    )
)

answer_runs = answer_runs[
    [
        (analysis_set, case_id) in current_keys
        for analysis_set, case_id in zip(
            answer_runs["analysis_set"],
            answer_runs["case_id"],
        )
    ]
].copy()

case_files = {
    "bioasq_train": PROCESSED_DIR / "bioasq_train_cases.jsonl",
    "bioasq_test": PROCESSED_DIR / "bioasq_test_cases.jsonl",
    "archehr_train": PROCESSED_DIR / "archehr_train_cases.jsonl",
    "archehr_test": PROCESSED_DIR / "archehr_test_cases.jsonl",
}

active_sets = set(analysis["analysis_set"])

case_by_key = {
    (analysis_set, case["case_id"]): case
    for analysis_set, path in case_files.items()
    if analysis_set in active_sets
    for case in load_jsonl(path)
}

print("Cases loaded:", len(analysis))
print(analysis["analysis_set"].value_counts())

Cases loaded: 993
analysis_set
bioasq_train     973
archehr_train     20
Name: count, dtype: int64


## 4. Select representative answers

The representative answer is taken from the largest semantic answer cluster from Notebook 03.

In [6]:
representative_rows = []

for (analysis_set, case_id), group in answer_runs.groupby(
    ["analysis_set", "case_id"],
    sort=False,
):
    row = representative_answer(group)
    row.update({
        "analysis_set": analysis_set,
        "case_id": case_id,
    })
    representative_rows.append(row)

representative_df = pd.DataFrame(representative_rows)

analysis = analysis.merge(
    representative_df,
    on=["analysis_set", "case_id"],
    how="left",
    validate="one_to_one",
)

if analysis["representative_answer"].isna().any():
    missing = analysis.loc[
        analysis["representative_answer"].isna(),
        ["analysis_set", "case_id"],
    ]
    raise ValueError(
        f"Missing representative answers:\n{missing}"
    )

display(
    analysis[
        [
            "analysis_set",
            "case_id",
            "evidence_uncertainty",
            "answer_uncertainty",
            "answer_pairwise_distance",
            "representative_cluster_size",
        ]
    ]
)

,analysis_set,case_id,evidence_uncertainty,answer_uncertainty,answer_pairwise_distance,representative_cluster_size
0,bioasq_train,bioasq13b_55031181e9bde69634000014,0.222176,-0.000000,0.046087,10
1,bioasq_train,bioasq13b_532f062ad6d3ac6a34000027,0.374005,0.141182,0.055142,9
2,bioasq_train,bioasq13b_54f49995d0d681a040000002,0.311111,-0.000000,0.002728,10
3,bioasq_train,bioasq13b_535d78137d100faa09000005,0.273505,-0.000000,0.053254,10
4,bioasq_train,bioasq13b_52d946c798d023950500000a,0.323552,-0.000000,0.031167,10
...,...,...,...,...,...,...
988,archehr_train,archehr_16,0.000000,-0.000000,0.007544,10
989,archehr_train,archehr_17,0.040000,0.447173,0.147417,5
990,archehr_train,archehr_18,0.405185,0.141182,0.060100,9
991,archehr_train,archehr_19,0.000000,0.277528,0.083709,8


## 5. Judge representative-answer correctness

Correctness is evaluated automatically against the expert reference answer(s). This is an evaluation proxy rather than clinical ground truth.

In [8]:
existing_judges = load_jsonl(JUDGE_PATH)

judge_lookup = {
    (
        row["analysis_set"],
        row["case_id"],
        row["representative_answer"],
    ): row
    for row in existing_judges
}

# Check whether any judge results are missing
missing_judges = [
    row
    for row in analysis.itertuples()
    if (
        row.analysis_set,
        row.case_id,
        row.representative_answer,
    ) not in judge_lookup
]

print("Loaded judge results:", len(judge_lookup))
print("Missing judge results:", len(missing_judges))

# Only run the judge model if results are missing
if missing_judges:

    try:
        check_judge_server()
    except Exception as exc:
        raise RuntimeError(
            f"Start {JUDGE_BACKEND} before running answer evaluation."
        ) from exc

    for row in missing_judges:

        case = case_by_key[
            (row.analysis_set, row.case_id)
        ]

        result = judge_answer(
            case["question"],
            case["reference_answers"],
            row.representative_answer,
        )

        record = {
            "analysis_set": row.analysis_set,
            "case_id": row.case_id,
            "representative_answer": row.representative_answer,
            "correct": bool(result["correct"]),
            "reason": result["reason"],
        }

        append_jsonl(record, JUDGE_PATH)

        key = (
            row.analysis_set,
            row.case_id,
            row.representative_answer,
        )

        judge_lookup[key] = record

        print(
            row.analysis_set,
            row.case_id,
            result["correct"],
        )


analysis["judge_correct"] = [
    judge_lookup[
        (
            row.analysis_set,
            row.case_id,
            row.representative_answer,
        )
    ]["correct"]
    for row in analysis.itertuples()
]

analysis["judge_reason"] = [
    judge_lookup[
        (
            row.analysis_set,
            row.case_id,
            row.representative_answer,
        )
    ]["reason"]
    for row in analysis.itertuples()
]

analysis["judge_correct"] = (
    analysis["judge_correct"].astype(int)
)

analysis["answer_error"] = (
    1 - analysis["judge_correct"]
)

Loaded judge results: 993
Missing judge results: 0


## 6. Combined analysis

In [31]:
# Both primary uncertainty measures are already normalised to 0-1.
analysis["combined_uncertainty"] = (
    analysis["evidence_uncertainty"]
    + analysis["answer_uncertainty"]
) / 2.0


# Joint behavioural profiles
# EU = 0: stable evidence selection
# EU > 0: unstable evidence selection
# AU = 0: stable answer generation
# AU > 0: variable answer generation

analysis["evidence_profile"] = np.where(
    analysis["evidence_uncertainty"] == 0,
    "Stable",
    "Unstable",
)

analysis["answer_profile"] = np.where(
    analysis["answer_uncertainty"] == 0,
    "Stable",
    "Variable",
)

analysis["joint_profile"] = (
    analysis["evidence_profile"]
    + "–"
    + analysis["answer_profile"]
)


# Save case-level analysis with profile labels
analysis.to_csv(
    CASE_PATH,
    index=False,
)


# Correlations
correlation_rows = []

correlation_pairs = [
    (
        "evidence_vs_answer_uncertainty",
        "evidence_uncertainty",
        "answer_uncertainty",
    ),
    (
        "evidence_uncertainty_vs_answer_error",
        "evidence_uncertainty",
        "answer_error",
    ),
    (
        "answer_uncertainty_vs_answer_error",
        "answer_uncertainty",
        "answer_error",
    ),
    (
        "combined_uncertainty_vs_answer_error",
        "combined_uncertainty",
        "answer_error",
    ),
    (
        "pairwise_distance_vs_answer_error",
        "answer_pairwise_distance",
        "answer_error",
    ),
    (
        "evidence_uncertainty_vs_evidence_recall",
        "evidence_uncertainty",
        "evidence_recall",
    ),
    (
        "evidence_uncertainty_vs_evidence_f1",
        "evidence_uncertainty",
        "evidence_f1",
    ),
]

for analysis_set, group in analysis.groupby(
    "analysis_set",
    sort=False,
):
    for relationship, x_col, y_col in correlation_pairs:
        rho, p_value, n = safe_spearman(
            group[x_col],
            group[y_col],
        )

        correlation_rows.append({
            "analysis_set": analysis_set,
            "relationship": relationship,
            "n": n,
            "spearman_rho": rho,
            "p_value": p_value,
        })

correlations = pd.DataFrame(correlation_rows)

correlations.to_csv(
    CORRELATION_PATH,
    index=False,
)


# Reliability prediction
primary_signals = [
    "evidence_uncertainty",
    "answer_uncertainty",
    "combined_uncertainty",
]

companion_signals = [
    "answer_pairwise_distance",
]

prediction_rows = []
risk_rows = []
selective_rows = []

for analysis_set, group in analysis.groupby(
    "analysis_set",
    sort=False,
):
    for signal in primary_signals + companion_signals:

        curve = compute_risk_coverage(
            group,
            signal,
        )

        for row in curve.itertuples():
            risk_rows.append({
                "analysis_set": analysis_set,
                "signal": signal,
                "coverage": row.coverage,
                "n_retained": row.n_retained,
                "retained_error_rate": row.retained_error_rate,
                "retained_correctness": row.retained_correctness,
            })

        prediction_rows.append({
            "analysis_set": analysis_set,
            "signal": signal,
            "signal_role": (
                "primary"
                if signal in primary_signals
                else "companion"
            ),
            "n_cases": len(group),
            "n_incorrect": int(
                group["answer_error"].sum()
            ),
            "incorrect_rate": (
                group["answer_error"].mean()
            ),
            "auroc": safe_auroc(
                group["answer_error"],
                group[signal],
            ),
            # Lower AURC is better.
            "aurc": (
                curve["retained_error_rate"].mean()
            ),
        })

    for signal in primary_signals:
        table = selective_review(
            group,
            signal,
        )

        for row in table.itertuples():
            selective_rows.append({
                "analysis_set": analysis_set,
                "signal": signal,
                "target_coverage": row.target_coverage,
                "coverage": row.coverage,
                "n_retained": row.n_retained,
                "retained_error_rate": row.retained_error_rate,
                "retained_correctness": row.retained_correctness,
            })


prediction_metrics = pd.DataFrame(
    prediction_rows
)

risk_coverage = pd.DataFrame(
    risk_rows
)

selective_review_table = pd.DataFrame(
    selective_rows
)

prediction_metrics.to_csv(
    PREDICTION_PATH,
    index=False,
)

risk_coverage.to_csv(
    RISK_PATH,
    index=False,
)

selective_review_table.to_csv(
    SELECTIVE_PATH,
    index=False,
)

# Joint profile summary
profile_order = [
    "Stable–Stable",
    "Stable–Variable",
    "Unstable–Stable",
    "Unstable–Variable",
]

analysis["joint_profile"] = pd.Categorical(
    analysis["joint_profile"],
    categories=profile_order,
    ordered=True,
)

profile_summary = (
    analysis
    .groupby(
        ["analysis_set", "joint_profile"],
        observed=False,
    )
    .agg(
        n=("case_id", "size"),
        correct_n=("judge_correct", "sum"),
        incorrect_n=("answer_error", "sum"),
        error_rate=("answer_error", "mean"),
        median_eu=("evidence_uncertainty", "median"),
        median_au=("answer_uncertainty", "median"),
    )
    .reset_index()
)

profile_summary["profile_percent"] = (
    profile_summary["n"]
    / profile_summary.groupby(
        "analysis_set"
    )["n"].transform("sum")
    * 100
)

profile_summary.to_csv(
    ANALYSIS_DIR / "joint_profile_summary.csv",
    index=False,
)


print("Saved:", ANALYSIS_DIR)

display(
    analysis.groupby("analysis_set")[
        [
            "evidence_uncertainty",
            "answer_uncertainty",
            "answer_pairwise_distance",
            "judge_correct",
        ]
    ].mean()
)

display(
    prediction_metrics[
        prediction_metrics["signal_role"]
        == "primary"
    ]
)

display(
    correlations[
        correlations["relationship"]
        == "evidence_vs_answer_uncertainty"
    ]
)

display(
    selective_review_table[
        selective_review_table["signal"]
        == "combined_uncertainty"
    ]
)

display(profile_summary)

Saved: /Users/sangbin/Desktop/Dissertation/Msc_Dissertation/outputs/combined_analysis/full


,evidence_uncertainty,answer_uncertainty,answer_pairwise_distance,judge_correct
analysis_set,,,,
archehr_train,0.065459,0.128494,0.051251,0.450000
bioasq_train,0.144070,0.018976,0.023966,0.873587


,analysis_set,signal,signal_role,n_cases,n_incorrect,incorrect_rate,auroc,aurc
0,bioasq_train,evidence_uncertainty,primary,973,123,0.126413,0.586102,0.101347
1,bioasq_train,answer_uncertainty,primary,973,123,0.126413,0.515275,0.123042
2,bioasq_train,combined_uncertainty,primary,973,123,0.126413,0.586992,0.100640
4,archehr_train,evidence_uncertainty,primary,20,11,0.550000,0.459596,0.574625
5,archehr_train,answer_uncertainty,primary,20,11,0.550000,0.419192,0.605768
6,archehr_train,combined_uncertainty,primary,20,11,0.550000,0.398990,0.611254


,analysis_set,relationship,n,spearman_rho,p_value
0,bioasq_train,evidence_vs_answer_uncertainty,973,0.117224,0.000248
7,archehr_train,evidence_vs_answer_uncertainty,20,0.201570,0.394101


,analysis_set,signal,target_coverage,coverage,n_retained,retained_error_rate,retained_correctness
10,bioasq_train,combined_uncertainty,1.0,1.000000,973,0.126413,0.873587
11,bioasq_train,combined_uncertainty,0.9,0.900308,876,0.118721,0.881279
12,bioasq_train,combined_uncertainty,0.8,0.800617,779,0.114249,0.885751
13,bioasq_train,combined_uncertainty,0.7,0.700925,682,0.102639,0.897361
14,bioasq_train,combined_uncertainty,0.6,0.600206,584,0.102740,0.897260
25,archehr_train,combined_uncertainty,1.0,1.000000,20,0.550000,0.450000
26,archehr_train,combined_uncertainty,0.9,0.900000,18,0.555556,0.444444
27,archehr_train,combined_uncertainty,0.8,0.800000,16,0.562500,0.437500
28,archehr_train,combined_uncertainty,0.7,0.700000,14,0.571429,0.428571
29,archehr_train,combined_uncertainty,0.6,0.600000,12,0.666667,0.333333


,analysis_set,joint_profile,n,correct_n,incorrect_n,error_rate,median_eu,median_au,profile_percent
0,archehr_train,Stable–Stable,8,3,5,0.625000,0.000000,-0.000000,40.000000
1,archehr_train,Stable–Variable,4,2,2,0.500000,0.000000,0.247425,20.000000
2,archehr_train,Unstable–Stable,3,1,2,0.666667,0.133333,-0.000000,15.000000
3,archehr_train,Unstable–Variable,5,3,2,0.400000,0.093407,0.217322,25.000000
4,bioasq_train,Stable–Stable,357,325,32,0.089636,0.000000,-0.000000,36.690647
5,bioasq_train,Stable–Variable,12,10,2,0.166667,0.000000,0.141182,1.233299
6,bioasq_train,Unstable–Stable,537,459,78,0.145251,0.209562,-0.000000,55.190134
7,bioasq_train,Unstable–Variable,67,56,11,0.164179,0.198122,0.217322,6.885920


## 7. Bootstrap 95% confidence intervals

Bootstrap 95% confidence intervals were used for the final RQ3 and RQ4 metrics.

In [19]:
# Calculate percentile-based 95% CI
def percentile_ci(values, alpha=0.05):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan

    return (
        np.quantile(values, alpha / 2),
        np.quantile(values, 1 - alpha / 2),
    )

# Calculate AURC from the risk-coverage curve
def aurc_score(df, signal):
    curve = compute_risk_coverage(df, signal)
    return curve["retained_error_rate"].mean()

# Resample correct and incorrect cases separately
def stratified_sample(df, rng):
    sampled = []

    for label in df["answer_error"].unique():
        idx = df.index[df["answer_error"] == label]
        sampled.extend(
            rng.choice(idx, size=len(idx), replace=True)
        )

    return df.loc[sampled].reset_index(drop=True)

# Bootstrap 95% CI for RQ3 Spearman correlation
def bootstrap_spearman(
    df,
    n_bootstrap=N_BOOTSTRAP,
    seed=BOOTSTRAP_SEED,
):     

    rng = np.random.default_rng(seed)

    rho, _, _ = safe_spearman(
        df["evidence_uncertainty"],
        df["answer_uncertainty"],
    )

    boot_rho = []

    for _ in range(n_bootstrap):
        sample = df.sample(
            n=len(df),
            replace=True,
            random_state=int(rng.integers(0, 1_000_000)),
        )

        r, _, _ = safe_spearman(
            sample["evidence_uncertainty"],
            sample["answer_uncertainty"],
        )

        if np.isfinite(r):
            boot_rho.append(r)

    lower, upper = percentile_ci(
        boot_rho,
        BOOTSTRAP_ALPHA,
    )

    return rho, lower, upper

# Paired stratified bootstrap for AUROC and AURC
def bootstrap_rq4(
    df,
    signals,
    n_bootstrap=N_BOOTSTRAP,
    seed=BOOTSTRAP_SEED,
):     

    point = {}

    for signal in signals:
        point[signal] = {
            "auroc": safe_auroc(
                df["answer_error"],
                df[signal],
            ),
            "aurc": aurc_score(df, signal),
        }

    boot = {
        signal: {"auroc": [], "aurc": []}
        for signal in signals
    }

    comparisons = {
        "combined_minus_evidence": {
            "auroc": [],
            "aurc": [],
        },
        "combined_minus_answer": {
            "auroc": [],
            "aurc": [],
        },
    }

    rng = np.random.default_rng(seed)

    for _ in range(n_bootstrap):
        sample = stratified_sample(df, rng)

        results = {}

        for signal in signals:
            auroc = safe_auroc(
                sample["answer_error"],
                sample[signal],
            )
            aurc = aurc_score(sample, signal)

            results[signal] = {
                "auroc": auroc,
                "aurc": aurc,
            }

            boot[signal]["auroc"].append(auroc)
            boot[signal]["aurc"].append(aurc)

        # Compare combined uncertainty with evidence uncertainty
        comparisons["combined_minus_evidence"]["auroc"].append(
            results["combined_uncertainty"]["auroc"]
            - results["evidence_uncertainty"]["auroc"]
        )

        comparisons["combined_minus_evidence"]["aurc"].append(
            results["combined_uncertainty"]["aurc"]
            - results["evidence_uncertainty"]["aurc"]
        )

        # Compare combined uncertainty with answer uncertainty
        comparisons["combined_minus_answer"]["auroc"].append(
            results["combined_uncertainty"]["auroc"]
            - results["answer_uncertainty"]["auroc"]
        )

        comparisons["combined_minus_answer"]["aurc"].append(
            results["combined_uncertainty"]["aurc"]
            - results["answer_uncertainty"]["aurc"]
        )

    metric_rows = []

    for signal in signals:
        for metric in ["auroc", "aurc"]:
            lower, upper = percentile_ci(
                boot[signal][metric],
                BOOTSTRAP_ALPHA,
            )

            metric_rows.append({
                "signal": signal,
                "metric": metric,
                "point_estimate": point[signal][metric],
                "ci_lower": lower,
                "ci_upper": upper,
            })

    comparison_rows = []

    comparison_pairs = {
        "combined_minus_evidence":
            ("combined_uncertainty", "evidence_uncertainty"),
        "combined_minus_answer":
            ("combined_uncertainty", "answer_uncertainty"),
    }

    for name, (combined, other) in comparison_pairs.items():
        for metric in ["auroc", "aurc"]:
            lower, upper = percentile_ci(
                comparisons[name][metric],
                BOOTSTRAP_ALPHA,
            )

            comparison_rows.append({
                "comparison": name,
                "metric": metric,
                "point_difference":
                    point[combined][metric]
                    - point[other][metric],
                "ci_lower": lower,
                "ci_upper": upper,
            })

    return metric_rows, comparison_rows


if RUN_BOOTSTRAP:

    bootstrap_metrics = []
    bootstrap_comparisons = []

    # Run bootstrap separately for each analysis dataset
    for i, (analysis_set, group) in enumerate(
        analysis.groupby("analysis_set", sort=False)
    ):
        group = group.reset_index(drop=True)
        seed = BOOTSTRAP_SEED + i

        # RQ3: relationship between evidence and answer uncertainty
        rho, rho_lower, rho_upper = bootstrap_spearman(
            group,
            seed=seed,
        )

        bootstrap_metrics.append({
            "analysis_set": analysis_set,
            "signal": "evidence_vs_answer_uncertainty",
            "metric": "spearman_rho",
            "point_estimate": rho,
            "ci_lower": rho_lower,
            "ci_upper": rho_upper,
        })

        # RQ4 requires both correct and incorrect cases
        if group["answer_error"].nunique() >= 2:

            metric_rows, comparison_rows = bootstrap_rq4(
                group,
                primary_signals,
                seed=seed + 1,
            )

            for row in metric_rows:
                row["analysis_set"] = analysis_set
                bootstrap_metrics.append(row)

            for row in comparison_rows:
                row["analysis_set"] = analysis_set
                bootstrap_comparisons.append(row)

    bootstrap_metrics = pd.DataFrame(bootstrap_metrics)
    bootstrap_comparisons = pd.DataFrame(
        bootstrap_comparisons
    )

    # Save bootstrap results
    bootstrap_metrics.to_csv(
        BOOTSTRAP_METRICS_PATH,
        index=False,
    )

    bootstrap_comparisons.to_csv(
        BOOTSTRAP_COMPARISONS_PATH,
        index=False,
    )

    display(bootstrap_metrics)
    display(bootstrap_comparisons)

else:
    print("Bootstrap skipped in SAMPLE_MODE.")

,analysis_set,signal,metric,point_estimate,ci_lower,ci_upper
0,bioasq_train,evidence_vs_answer_uncertainty,spearman_rho,0.117224,0.060497,0.174011
1,bioasq_train,evidence_uncertainty,auroc,0.586102,0.534385,0.637822
2,bioasq_train,evidence_uncertainty,aurc,0.101347,0.085650,0.116991
3,bioasq_train,answer_uncertainty,auroc,0.515275,0.487230,0.546153
4,bioasq_train,answer_uncertainty,aurc,0.123042,0.116018,0.129423
5,bioasq_train,combined_uncertainty,auroc,0.586992,0.535088,0.638085
6,bioasq_train,combined_uncertainty,aurc,0.100640,0.084865,0.116005
7,archehr_train,evidence_vs_answer_uncertainty,spearman_rho,0.201570,-0.222852,0.618000
8,archehr_train,evidence_uncertainty,auroc,0.459596,0.232323,0.686869
9,archehr_train,evidence_uncertainty,aurc,0.574625,0.441104,0.715615


,comparison,metric,point_difference,ci_lower,ci_upper,analysis_set
0,combined_minus_evidence,auroc,0.000890,-0.014333,0.018501,bioasq_train
1,combined_minus_evidence,aurc,-0.000707,-0.005726,0.003125,bioasq_train
2,combined_minus_answer,auroc,0.071717,0.023430,0.120911,bioasq_train
3,combined_minus_answer,aurc,-0.022402,-0.037578,-0.007389,bioasq_train
4,combined_minus_evidence,auroc,-0.060606,-0.282828,0.141414,archehr_train
5,combined_minus_evidence,aurc,0.036629,-0.085746,0.174817,archehr_train
6,combined_minus_answer,auroc,-0.020202,-0.171717,0.116162,archehr_train
7,combined_minus_answer,aurc,0.005486,-0.084288,0.110583,archehr_train


## 8. Reviews

#### 1) Manual review of automated correctness judgements

Manual review to compare a subset of the correctness judgements by LLM with manual assessment.

For ArchEHR-QA development cases, all 20  were included and for BioASQ, 10 cases
judged correct and 10 cases judged incorrect were randomly selected.

In [25]:
REVIEW_SEED = 42
REVIEW_PATH = ANALYSIS_DIR / "manual_judge_review.csv"

if REVIEW_PATH.exists():
    print("Manual review file already exists:")
    print(REVIEW_PATH)

else:
    # ArchEHR train dataset
    arch_review = analysis[
        analysis["analysis_set"] == "archehr_train"
    ].copy()
    # All 20 cases
    assert len(arch_review) == 20

    # BioASQ train dataset
    bio = analysis[
        analysis["analysis_set"] == "bioasq_train"
    ].copy()
    # 10 judged correct cases
    bio_correct = bio[
        bio["judge_correct"] == 1
    ].sample(
        n=10,
        random_state=REVIEW_SEED,
    )
    # 10 judged incorrect cases
    bio_incorrect = bio[
        bio["judge_correct"] == 0
    ].sample(
        n=10,
        random_state=REVIEW_SEED,
    )

    # Concatenate all the cases 
    review = pd.concat(
        [
            arch_review,
            bio_correct,
            bio_incorrect,
        ],
        ignore_index=True,
    )

    # Shuffle order so BioASQ correct/incorrect cases are not grouped together
    review = review.sample(
        frac=1,
        random_state=REVIEW_SEED,
    ).reset_index(drop=True)

    review["review_id"] = [
        f"R{i:02d}"
        for i in range(1, len(review) + 1)
    ]

    # Add question and reference answer
    questions = []
    references = []

    for row in review.itertuples():
        case = case_by_key[
            (row.analysis_set, row.case_id)
        ]

        questions.append(
            case["question"]
        )

        references.append(
            "\n\n".join(
                case["reference_answers"]
            )
        )

    review["question"] = questions
    review["reference_answers"] = references

    # Keep only information needed for blinded review
    manual_review = review[
        [
            "review_id",
            "analysis_set",
            "case_id",
            "question",
            "reference_answers",
            "representative_answer",
        ]
    ].copy()

    manual_review["manual_label"] = ""

    manual_review.to_csv(
        REVIEW_PATH,
        index=False,
    )

    print("Saved:", REVIEW_PATH)

Manual review file already exists:
/Users/sangbin/Desktop/Dissertation/Msc_Dissertation/outputs/combined_analysis/full/manual_judge_review.csv


In [30]:
# Compare manual review with LLM judge result
manual_review = pd.read_csv(
    REVIEW_PATH
)

manual_review["manual_label"] = (
    manual_review["manual_label"]
    .fillna("")
    .str.strip()
)

# Check that manual review is complete
allowed_labels = {
    "Correct",
    "Incorrect",
    "Unclear",
}

invalid = manual_review[
    ~manual_review["manual_label"].isin(
        allowed_labels
    )
]

if len(invalid) > 0:
    display(
        invalid[
            [
                "review_id",
                "manual_label",
            ]
        ]
    )

    raise ValueError(
        "Use only Correct, Incorrect, or Unclear."
    )

# Add LLM judge result
judge_results = analysis[
    [
        "analysis_set",
        "case_id",
        "representative_answer",
        "judge_correct",
        "judge_reason",
    ]
].copy()

comparison = manual_review.merge(
    judge_results,
    on=[
        "analysis_set",
        "case_id",
        "representative_answer",
    ],
    how="left",
    validate="one_to_one",
)

comparison["judge_label"] = (
    comparison["judge_correct"]
    .astype(int)
    .map({
        1: "Correct",
        0: "Incorrect",
    })
)

# Compare labels
comparison["comparison"] = [
    (
        "Manual unclear"
        if manual == "Unclear"
        else "Agree"
        if manual == judge
        else "Disagree"
    )
    for manual, judge in zip(
        comparison["manual_label"],
        comparison["judge_label"],
    )
]

print(
    comparison[
        "comparison"
    ].value_counts()
)

display(
    comparison.groupby(
        [
            "analysis_set",
            "comparison",
        ]
    )
    .size()
    .rename("n")
    .reset_index()
)

# Check for disagreements only
disagreements = comparison[
    comparison["comparison"] == "Disagree"
].copy()

print(
    "Disagreements:",
    len(disagreements),
)

display(
    disagreements[
        [
            "review_id",
            "analysis_set",
            "case_id",
            "manual_label",
            "judge_label",
            "judge_reason",
        ]
    ]
)

comparison
Agree             25
Disagree          12
Manual unclear     3
Name: count, dtype: int64


,analysis_set,comparison,n
0,archehr_train,Agree,13
1,archehr_train,Disagree,5
2,archehr_train,Manual unclear,2
3,bioasq_train,Agree,12
4,bioasq_train,Disagree,7
5,bioasq_train,Manual unclear,1


Disagreements: 12


,review_id,analysis_set,case_id,manual_label,judge_label,judge_reason
1,R02,archehr_train,archehr_17,Correct,Incorrect,The generated answer does not address relievin...
6,R07,bioasq_train,bioasq13b_623648513a8413c6530000ae,Correct,Incorrect,The generated answer inaccurately limits AIS t...
8,R09,bioasq_train,bioasq13b_5a43a933966455904c00000b,Correct,Incorrect,unsupported claim about oxidation derivatives
10,R11,bioasq_train,bioasq13b_56a3a6c9496b62f23f000008,Incorrect,Correct,The generated answer accurately describes TFBS...
12,R13,archehr_train,archehr_14,Correct,Incorrect,The generated answer does not directly address...
19,R20,bioasq_train,bioasq13b_606b61f794d57fd879000068,Correct,Incorrect,The generated answer claims that the downstrea...
20,R21,archehr_train,archehr_6,Correct,Incorrect,generated answer contains unsupported claims a...
25,R26,archehr_train,archehr_3,Correct,Incorrect,omitted major parts of the expected course of ...
26,R27,bioasq_train,bioasq13b_530cf22aa177c6630c000001,Correct,Incorrect,generated answer introduces unsupported claims...
27,R28,bioasq_train,bioasq13b_5c6585097c78d69471000003,Correct,Incorrect,unsupported claims about regulating cardiac ge...


In [29]:
disagreement_review = disagreements.copy()

questions = []
references = []

for row in disagreement_review.itertuples():
    case = case_by_key[
        (row.analysis_set, row.case_id)
    ]

    questions.append(case["question"])
    references.append(
        "\n\n".join(case["reference_answers"])
    )

disagreement_review["question"] = questions
disagreement_review["reference_answers"] = references

display(
    disagreement_review[
        [
            "review_id",
            "analysis_set",
            "case_id",
            "question",
            "reference_answers",
            "representative_answer",
            "manual_label",
            "judge_label",
            "judge_reason",
        ]
    ]
)

,review_id,analysis_set,case_id,question,reference_answers,representative_answer,manual_label,judge_label,judge_reason
1,R02,archehr_train,archehr_17,What should he do to relieve palpitations and ...,Palpitations may occur if sleep apnea is untre...,The patient was started on levothyroxine and s...,Correct,Incorrect,The generated answer does not address relievin...
6,R07,bioasq_train,bioasq13b_623648513a8413c6530000ae,What is Abbreviated Injury Scale (AIS) used to...,The Abbreviated Injury Scale (AIS) is an objec...,The Abbreviated Injury Scale (AIS) is used to ...,Correct,Incorrect,The generated answer inaccurately limits AIS t...
8,R09,bioasq_train,bioasq13b_5a43a933966455904c00000b,How may CTCF mediate splicing?,Two different mechanisms convey DNA methylatio...,CTCF mediates splicing by promoting inclusion ...,Correct,Incorrect,unsupported claim about oxidation derivatives
10,R11,bioasq_train,bioasq13b_56a3a6c9496b62f23f000008,What is TFBSshape?,To utilize DNA shape information when analysin...,TFBSshape is a motif database for DNA shape fe...,Incorrect,Correct,The generated answer accurately describes TFBS...
12,R13,archehr_train,archehr_14,Was there any evidence for stomach cancer?,The patient does not have any evidence of stom...,The evidence does not address stomach cancer. ...,Correct,Incorrect,The generated answer does not directly address...
19,R20,bioasq_train,bioasq13b_606b61f794d57fd879000068,What is known about natriuretic peptide recept...,Atrial natriuretic peptide (ANP) and its natri...,"Natriuretic peptide receptor A (NPRA), also kn...",Correct,Incorrect,The generated answer claims that the downstrea...
20,R21,archehr_train,archehr_6,Why did they find out later that he had fungal...,Preliminary tests showed Candida infection in ...,The evidence indicates that *Torulopsis glabra...,Correct,Incorrect,generated answer contains unsupported claims a...
25,R26,archehr_train,archehr_3,What is the expected course of recovery for him?,"This patient should expect to have drowsiness,...","Some symptoms following a head injury, such as...",Correct,Incorrect,omitted major parts of the expected course of ...
26,R27,bioasq_train,bioasq13b_530cf22aa177c6630c000001,What is the role of Inn1 in cytokinesis?,Inn1 associates with the contractile actomyosi...,Inn1 plays a role in cytokinesis by regulating...,Correct,Incorrect,generated answer introduces unsupported claims...
27,R28,bioasq_train,bioasq13b_5c6585097c78d69471000003,What is the function of the Nup153 protein?,Nup153 is a large (153 kD) O-linked glyco-prot...,Nup153 plays pivotal roles in nuclear pore fun...,Correct,Incorrect,unsupported claims about regulating cardiac ge...


#### 2) Exploratory Case-Level Review

Exploratory case-level review to examine the evidence and answer patterns underlying selected joint behavioural profiles.

In [39]:
heldout = pd.read_csv(HELDOUT_PATH)

profiles = [
    "Stable–Stable",
    "Stable–Variable",
    "Unstable–Stable",
    "Unstable–Variable",
]

selected_cases = []

for profile in profiles:
    group = heldout[
        heldout["joint_profile"] == profile
    ]

    sampled = group.sample(
        n=1,
        random_state=42,
    )

    selected_cases.append(sampled)

selected_cases = pd.concat(
    selected_cases,
    ignore_index=True,
)

display(
    selected_cases[
        [
            "case_id",
            "joint_profile",
            "evidence_uncertainty",
            "answer_uncertainty",
            "judge_correct",
        ]
    ]
)

,case_id,joint_profile,evidence_uncertainty,answer_uncertainty,judge_correct
0,archehr_66,Stable–Stable,0.000000,-0.000000,1
1,archehr_23,Stable–Variable,0.000000,0.217322,1
2,archehr_52,Unstable–Stable,0.177778,-0.000000,0
3,archehr_88,Unstable–Variable,0.084040,0.518352,0


In [43]:
selected_ids = selected_cases["case_id"].tolist()

evidence_runs = pd.DataFrame(
    load_jsonl(HELDOUT_DIR / "evidence_runs.jsonl")
)

evidence_summary = pd.read_csv(
    HELDOUT_DIR / "evidence_summary.csv"
)

answer_runs = pd.read_csv(
    HELDOUT_DIR / "answer_runs_with_clusters.csv"
)

judge_results = pd.DataFrame(
    load_jsonl(HELDOUT_DIR / "judge_results.jsonl")
)

cases = load_jsonl(
    PROCESSED_DIR / "archehr_test_cases.jsonl"
)

case_lookup = {
    case["case_id"]: case
    for case in cases
}

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)


for case_id in selected_ids:

    case = case_lookup[case_id]

    print("=="*5)
    print(case_id)
    print("=="*5)

    print("\n[QUESTION]")
    print(case["question"])

    print("\n[REFERENCE ANSWER]")
    print(case["reference_answers"][0])

    print("\n[EVIDENCE SELECTION RUNS]")
    display(
        evidence_runs[
            evidence_runs["case_id"] == case_id
        ][
            [
                "run_id",
                "selected_sentence_ids",
            ]
        ]
    )

    summary = evidence_summary[
        evidence_summary["case_id"] == case_id
    ].iloc[0]

    representative_ids = json.loads(
        summary["selected_sentence_ids"]
    )

    print("\n[REPRESENTATIVE EVIDENCE]")
    for sentence in case["sentences"]:
        if str(sentence["sentence_id"]) in set(
            map(str, representative_ids)
        ):
            print(
                f'[{sentence["sentence_id"]}] '
                f'{sentence["text"]}'
            )

    print("\n[ANSWER RUNS]")
    display(
        answer_runs[
            answer_runs["case_id"] == case_id
        ][
            [
                "run_id",
                "cluster_label",
                "answer",
            ]
        ]
    )

    print("\n[LLM JUDGE]")
    display(
        judge_results[
            judge_results["case_id"] == case_id
        ][
            [
                "correct",
                "representative_answer",
                "reason",
            ]
        ]
    )

archehr_66

[QUESTION]
How were the goals of care established?

[REFERENCE ANSWER]
Patient with squamous cell cervical cancer, s/p combined chemoradiation with nephrostomy tube presented with dark stools and drop in her Hct. Patient found to have new pulmonary metastases and early cardiac tamponade. During a meeting with her family and social work, the patient made the decision that she did not want to undergo any more chemotherapy or invasive therapies including pericardiocentesis. The patient will be discharged to hospice with a focus on comfort.

[EVIDENCE SELECTION RUNS]


,run_id,selected_sentence_ids
450,0,"[3, 7, 9, 11, 17]"
451,1,"[3, 7, 9, 11, 17]"
452,2,"[3, 7, 9, 11, 17]"
453,3,"[3, 7, 9, 11, 17]"
454,4,"[3, 7, 9, 11, 17]"
455,5,"[3, 7, 9, 11, 17]"
456,6,"[3, 7, 9, 11, 17]"
457,7,"[3, 7, 9, 11, 17]"
458,8,"[3, 7, 9, 11, 17]"
459,9,"[3, 7, 9, 11, 17]"



[REPRESENTATIVE EVIDENCE]
[3] # Goals of care discussion - After it was determined that the patient had signs of cardiac tamponade, family meetings were held with the patient, her husband, daughters, and palliative care.
[7] Patient was never HD unstable - this finding prompted goals of care discussion as above given need of pericardiocentesis and then window if treatment desired.
[9] There was an initial plan to attempt further laser therapy, but this was decided against after the goals of care discussions.
[11] The patient stated she would not want to undergo further chemotherapy.
[17] After discussion with you and your family, it was decided to focus on treating your symptoms and alleviating suffering.

[ANSWER RUNS]


,run_id,cluster_label,answer
450,0,0,"Goals of care were established after signs of cardiac tamponade were noted and family meetings were held with the patient, her husband, and daughters, including palliative care. Further laser therapy and chemotherapy were decided against following these discussions, and the focus shifted to symptom management and alleviating suffering."
451,1,0,"Goals of care were established after it was determined the patient had signs of cardiac tamponade and prompted family meetings involving the patient, her husband, and daughters, with palliative care. Further laser therapy and chemotherapy were ruled out following these discussions, and the decision was made to focus on symptom management and alleviating suffering."
452,2,0,"Goals of care were established after signs of cardiac tamponade were noted and discussions were held with the patient, her husband, and daughters, including palliative care. The decision to focus on symptom management and alleviating suffering was made after discussions with the patient's family, and further laser therapy and chemotherapy were ruled out."
453,3,0,"Goals of care were established after signs of cardiac tamponade were noted and discussions were held with the patient, her husband, and daughters, including palliative care. Further laser therapy and chemotherapy were decided against following these discussions, and the focus shifted to symptom management and alleviating suffering."
454,4,0,"Goals of care were established after signs of cardiac tamponade were noted and family meetings were held with the patient, her husband, and daughters, including palliative care. Further laser therapy and chemotherapy were decided against following these discussions, and the focus shifted to symptom management and alleviating suffering."
455,5,0,"Goals of care were established after signs of cardiac tamponade were noted and discussions were held with the patient, her husband, and daughters, including palliative care. Further laser therapy and chemotherapy were ruled out following these discussions, and the decision was made to focus on symptom management and alleviating suffering."
456,6,0,"Goals of care were established after it was determined the patient had signs of cardiac tamponade and prompted family meetings involving the patient, her husband, and daughters, as well as palliative care. Further laser therapy and chemotherapy were decided against following these discussions, and the focus shifted to symptom management and alleviating suffering."
457,7,0,"Goals of care were established after signs of cardiac tamponade were noted and discussions were held with the patient, her husband, and daughters, including palliative care. Initially, further laser therapy was considered, but this was decided against following these discussions. Ultimately, the focus shifted to symptom management and alleviating suffering."
458,8,0,"Goals of care were established after signs of cardiac tamponade were noted and discussions were held with the patient, her husband, and daughters, including palliative care. Further laser therapy and chemotherapy were ruled out following these discussions, and it was decided to focus on symptom management and alleviating suffering."
459,9,0,"Goals of care were established after signs of cardiac tamponade were noted, prompting discussions with the patient, her husband, and daughters, involving palliative care. Initially, further laser therapy was considered, but this was abandoned following these discussions. Ultimately, the decision was made to focus on symptom management and alleviating suffering, aligning with the patient's desire to forgo further chemotherapy."



[LLM JUDGE]


...

,run_id,selected_sentence_ids
670,0,"[4, 5, 6, 26, 27, 28, 29, 30, 32, 41]"
671,1,"[2, 4, 5, 6, 26, 27, 28, 29, 30, 32]"
672,2,"[2, 4, 5, 6, 26, 27, 28, 29, 30, 32]"
673,3,"[2, 4, 5, 6, 26, 27, 28, 29, 30, 32]"
674,4,"[2, 4, 5, 6, 26, 27, 28, 29, 30, 32]"
675,5,"[2, 4, 5, 6, 26, 27, 28, 29, 30, 32]"
676,6,"[4, 5, 6, 26, 27, 28, 29, 30, 32, 41]"
677,7,"[4, 5, 6, 26, 27, 28, 29, 30, 32]"
678,8,"[2, 4, 5, 6, 26, 27, 28, 29, 30, 32]"
679,9,"[4, 5, 6, 26, 27, 28, 29, 30, 32]"



[REPRESENTATIVE EVIDENCE]
[2] # Bacteremia/Abscesses/Antibiotic therapy
[4] CBC with noted leukopenia on [**8-5**] with progression to neutropenia [**8-8**].
[5] Out of concern that vancomycin could be contributing to decreasing counts, vanc d/c'ed and IV daptomycin started.
[6] Per ID will plan to treat MSSA bacteremia/abscesses with IV daptomycin to total of 6-8wk course.
[26] Upon imaging it was found that you had several abscesses throughout your spine as well as in your right knee.
[27] You underwent several surgeries to evacuate these infection and you were also started on antibiotics.
[28] Your antibiotic regimen was altered several times due to significant side effects.
[29] You experienced a rash after Nafcillin, and Vancomyocin was thought to contribute to low cell counts.
[30] You were placed on Daptomycin without side effect for a planned 8week course.
[32] You will receive repeat imaging after completion of your antibiotic therapy for interval evaluation of infection.

[A

,run_id,cluster_label,answer
670,0,1,"The patient is receiving a 6-8 week course of IV daptomycin to treat MSSA bacteremia and abscesses in the spine and right knee, following surgical evacuation of the abscesses. Repeat imaging will be performed after completion of antibiotic therapy to evaluate the infection. The patient previously experienced leukopenia and neutropenia, prompting a change in antibiotic regimen."
671,1,2,"Your mother's recovery will involve repeat imaging after completion of a planned 8-week course of IV daptomycin to treat MSSA bacteremia/abscesses. These abscesses were found throughout her spine and right knee, and she underwent surgeries to evacuate the infections. Her antibiotic regimen was altered several times due to side effects."
672,2,1,"The patient is receiving a 6-8 week course of IV daptomycin to treat MSSA bacteremia/abscesses in the spine and knee. Repeat imaging will be performed after completion of antibiotic therapy to evaluate the infection. Previous antibiotics, including nafcillin and vancomycin, were discontinued due to adverse effects."
673,3,1,"The patient is receiving a 6-8 week course of IV daptomycin to treat MSSA bacteremia and abscesses in the spine and knee. Repeat imaging will be performed after completion of antibiotic therapy to evaluate the infection. Her antibiotic regimen has been altered due to side effects, including a rash with nafcillin and low cell counts with vancomycin."
674,4,0,"Your current treatment plan involves a 6-8 week course of IV daptomycin for MSSA bacteremia/abscesses. Repeat imaging will be performed after completion of antibiotic therapy to evaluate the infection. You previously experienced leukopenia and neutropenia, and vancomycin was discontinued due to concerns about low cell counts."
675,5,3,Your current treatment plan involves a 6-8 week course of IV daptomycin for MSSA bacteremia/abscesses. Repeat imaging will be performed after completion of antibiotic therapy to evaluate the infection. The cause of your continued fever is not specified in the provided information.
676,6,0,Your current course of recovery involves a planned 6-8 week course of IV daptomycin to treat MSSA bacteremia and abscesses throughout your spine and right knee. Repeat imaging will be performed after completion of antibiotic therapy to evaluate the infection.
677,7,0,"Your current course of recovery involves a planned 6-8 week course of IV daptomycin to treat MSSA bacteremia and abscesses throughout your spine and right knee. Repeat imaging will be performed after completion of antibiotic therapy to evaluate the infection. You previously experienced leukopenia and neutropenia, prompting a change in antibiotics."
678,8,0,"Your current course of recovery involves a planned 6-8 week course of IV daptomycin to treat MSSA bacteremia/abscesses. Repeat imaging will be performed after completion of antibiotic therapy to evaluate the infection. You previously experienced leukopenia and neutropenia, and antibiotic regimens were altered due to side effects."
679,9,1,"The patient is receiving a 6-8 week course of IV daptomycin to treat MSSA bacteremia and abscesses in the spine and knee, following surgeries to evacuate the infections. Repeat imaging will be performed after completion of antibiotic therapy to evaluate for interval infection. The patient previously experienced leukopenia and neutropenia, and vancomycin was discontinued due to concerns about contributing to low cell counts."



[LLM JUDGE]


,correct,representative_answer,reason
67,False,"The patient is receiving a 6-8 week course of IV daptomycin to treat MSSA bacteremia and abscesses in the spine and right knee, following surgical evacuation of the abscesses. Repeat imaging will be performed after completion of antibiotic therapy to evaluate the infection. The patient previously experienced leukopenia and neutropenia, prompting a change in antibiotic regimen.",inconsistent information and omission of key details
